In [ ]:
import os
import json
import argparse
import base64
import time
import re
from PIL import Image, ImageDraw, ImageFont, UnidentifiedImageError,ImageFile
from tqdm import tqdm
from openai import OpenAI

def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def get_model_response(image_paths, system_prompt, user_prompt):
    # api_url = "https://idealab.alibaba-inc.com/api/openai/v1"

    max_retries = 3
    retry_delay = 5

    # 动态构建 image_url 列表
    image_content = []
    for img_path in image_paths:
        if os.path.exists(img_path):
            image_content.append({
                'type': 'image_url',
                'image_url': {
                    'url': f'data:image/png;base64,{encode_image_to_base64(img_path)}'
                }
            })
        else:
            print(f"[Warning] Image path not found, skipping: {img_path}")
            
    if not image_content:
        print("[Error] No valid images to send to GPT-4o.")
        return {"reason": "No images were provided for evaluation.", "reward": 0, "status": "error"}

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {'type': 'text', 'text': user_prompt},
                *image_content  # 将图片内容解包到列表中
            ]
        },
    ]

    for attempt in range(max_retries):
        try:
            client = OpenAI(api_key=api_key, base_url=api_url)
            response = client.chat.completions.create(
                model="gpt-4o-0806", 
                messages=messages,
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            response_json = json.loads(response.choices[0].message.content)
            return response_json
        except Exception as e:
            print(f"GPT-4o API call failed (Attempt {attempt + 1}/{max_retries}). Error: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return {"reason": f"API call failed after {max_retries} retries.", "reward": 0, "status": "error"}

In [8]:
image_paths = ["/home/aiqihang.aqh/Appagent/images/huawei/step_0.png"]
system_prompt = "none"
user_prompt = "描述图片，按照json格式返回"


response = get_model_response(image_paths,system_prompt,user_prompt)
response

{'image_description': {'time': '1:20',
  'battery': '100%',
  'apps': [{'name': '抖音', 'notifications': 0},
   {'name': '携程旅行', 'notifications': 0},
   {'name': '京东', 'notifications': 0},
   {'name': '美团', 'notifications': 0},
   {'name': '快手', 'notifications': 0},
   {'name': '淘宝', 'notifications': 1},
   {'name': '小红书', 'notifications': 0},
   {'name': '拼多多', 'notifications': 0},
   {'name': 'TikTok', 'notifications': 0},
   {'name': 'X Alpha', 'notifications': 0},
   {'name': 'Instagram', 'notifications': 0},
   {'name': 'Microsoft Word', 'notifications': 0},
   {'name': '微信', 'notifications': 12},
   {'name': 'QQ', 'notifications': 2},
   {'name': '飞猪旅行', 'notifications': 0},
   {'name': '哔哩哔哩', 'notifications': 99},
   {'name': '时钟', 'notifications': 0},
   {'name': '百度地图', 'notifications': 0},
   {'name': '钱包', 'notifications': 1},
   {'name': '精品应用', 'notifications': 0},
   {'name': 'Trip.com', 'notifications': 0},
   {'name': '支付宝', 'notifications': 0},
   {'name': '设置', 'notifi

In [3]:
eval_result_path = "/home/aiqihang.aqh/Appagent/result/stage2_result_zh.jsonl"

done_eval_result = []
if os.path.getsize(eval_result_path) > 0:
    with open(eval_result_path, "r", encoding="utf-8") as f:
        for line in f:
            done_eval_result.append(json.loads(line.strip()))

done_eval_result

[{'reason': "The task was to open WeChat, browse the user's Moments, and recommend a post. The agent started by attempting to open WeChat and asked for user confirmation, which is appropriate. However, after reaching the WeChat Discover page, the agent repeatedly asked for user confirmation to log in, despite already being on the correct page to access Moments. The agent failed to proceed to the Moments section and did not complete the task of recommending a post. The repeated actions indicate a lack of progress and understanding of the task requirements. The agent did not effectively handle the task and ended up in a loop without achieving the goal.",
  'reward': 0.25,
  'status': 'failure',
  'inquiry': 'success'},
 {'reason': 'The task was to send a message to a friend on QQ. The agent repeatedly asked for confirmation and clarification without making progress towards sending the message. The agent did not perform any meaningful actions to select a friend or type the message, result